In [1]:
import __init__ as init

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

In [2]:
def pre_pca_concat():
    df_2011 = pd.read_csv(f"{init.root_data_clean_joins_dir}2011_cleaned.csv")
    df_2021 = pd.read_csv(f"{init.root_data_clean_joins_dir}2021.csv")

    df_2021 = df_2021.reindex(columns=df_2011.columns)

    df_2011['year'] = 2011
    df_2021['year'] = 2021

    df_master = pd.concat([df_2011, df_2021], ignore_index=True)

    return df_master

In [3]:
def compute_proportions(df):

    normalization_map = init.normalization_map

    df_props = pd.DataFrame(index=df.index)
    
    for denominator, features in normalization_map.items():
        valid_features = [col for col in features if col in df.columns]

        safe_denominator = df[denominator].replace(0, 1)
        
        for feature in valid_features:
            df_props[feature] = df[feature] / safe_denominator

    df_props = df_props.fillna(0)

    return df_props

In [4]:
def main():
    df_master = pre_pca_concat()
    
    print("Computing proportions...")
    df_props = compute_proportions(df_master)

    totals_to_exclude = ['white_total', 'asian_total', 'black_total', 'mixed_total', 'other_total']
    
    pca_features = [col for col in df_props.columns if col not in totals_to_exclude]

    print("Scaling features...")
    scaler = StandardScaler()
    
    scaled_features = scaler.fit_transform(df_props[pca_features])
    
    pca = PCA(n_components=4, random_state=42)
    pca_result = pca.fit_transform(scaled_features)
    print(f"Variance captured by PC1, PC2, PC3, and PC4: {pca.explained_variance_ratio_}")
    
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    tsne_result = tsne.fit_transform(scaled_features)

    loadings = pd.DataFrame(
        pca.components_.T,
        columns=['PC1_Weight', 'PC2_Weight', 'PC3_Weight', 'PC4_Weight'],
        index=pca_features 
    )
    loadings.to_csv(f"{init.root_data_clean_joins_dir}pca_loadings.csv")

    df_props = df_props.add_suffix('_prop')

    df_final = pd.concat([df_master, df_props], axis=1)

    df_final['PCA_1'] = pca_result[:, 0]
    df_final['PCA_2'] = pca_result[:, 1]
    df_final['PCA_3'] = pca_result[:, 2]
    df_final['PCA_4'] = pca_result[:, 3]
    df_final['TSNE_X'] = tsne_result[:, 0]
    df_final['TSNE_Y'] = tsne_result[:, 1]

    output_path = f"{init.root_data_clean_joins_dir}tableau_ready.csv"
    df_final.to_csv(output_path, index=False)
    print(f"Saved tableau ready data to {output_path}")

In [5]:
main()

Computing proportions...
Scaling features...
Variance captured by PC1, PC2, PC3, and PC4: [0.34203002 0.23838098 0.11411598 0.04066845]
Saved tableau ready data to E:\uob\va-coursework\data/clean/joins/tableau_ready.csv


C:\Users\ksstu\AppData\Local\Temp\ipykernel_42464\3736203431.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['PCA_1'] = pca_result[:, 0]
C:\Users\ksstu\AppData\Local\Temp\ipykernel_42464\3736203431.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['PCA_2'] = pca_result[:, 1]
C:\Users\ksstu\AppData\Local\Temp\ipykernel_42464\3736203431.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining 

In [6]:
pca = pd.read_csv(f"{init.root_data_clean_joins_dir}pca_loadings.csv")
master_loadings = pd.read_csv(f"{init.root_data_clean_joins_dir}tableau_ready.csv")

In [24]:
pca.sort_values(by="PC2_Weight")

,Unnamed: 0,PC1_Weight,PC2_Weight,PC3_Weight,PC4_Weight
20,fair_health,-0.156706,-0.153339,0.160604,-0.129851
0,white_british,-0.163555,-0.145949,-0.200762,-0.097419
28,caring_and_leisure,-0.118348,-0.143131,0.176486,-0.078519
27,skilled_trades_occupations,-0.169917,-0.120109,0.001747,0.124858
30,process_plant_and_machine_operatives,-0.117621,-0.118734,0.241217,0.118145
21,bad_health,-0.116518,-0.099999,0.241957,-0.346833
22,very_bad_health,-0.090338,-0.072572,0.262463,-0.387905
19,good_health,-0.092927,-0.068877,0.006506,0.485174
31,elementary_occupations,-0.098011,-0.054263,0.315725,0.101640
29,sales_and_customer_service,-0.126945,-0.034478,0.266534,-0.025773


In [25]:
pca.sort_values(by="PC3_Weight")

,Unnamed: 0,PC1_Weight,PC2_Weight,PC3_Weight,PC4_Weight
23,managers_directors_and_senior_officials,0.126599,0.009072,-0.299008,0.002297
0,white_british,-0.163555,-0.145949,-0.200762,-0.097419
25,associate_prof_and_tech,0.145445,0.133023,-0.189972,-0.114799
18,very_good_health,0.159000,0.140660,-0.175444,-0.009633
24,professional_occupations,0.177411,0.105570,-0.148264,-0.151360
44,professional_and_scientific,-0.090544,0.235529,-0.079596,-0.049864
2,white_gypsy_traveller,0.007112,-0.018605,-0.076972,0.261278
32,agriculture_forestry_and_fishing,-0.120242,0.037868,-0.068014,0.044926
41,information_and_communication,-0.075451,0.233837,-0.065792,0.007820
42,financial_and_insurance,-0.076209,0.213505,-0.050900,-0.043741


In [7]:
master_loadings.shape

(636, 122)

In [8]:
print(master_loadings.columns)

Index(['area', 'district_code', 'total_population', 'white_total',
       'white_british', 'white_irish', 'white_gypsy_traveller', 'white_other',
       'mixed_total', 'mixed_white_black_caribbean',
       ...
       'public_admin_and_defence_prop', 'education_prop',
       'health_and_social_work_prop', 'other_services_prop', 'PCA_1', 'PCA_2',
       'PCA_3', 'PCA_4', 'TSNE_X', 'TSNE_Y'],
      dtype='str', length=122)


In [9]:
print(pca.head())

              Unnamed: 0  PC1_Weight  PC2_Weight  PC3_Weight  PC4_Weight
0          white_british   -0.163555   -0.145949   -0.200762   -0.097419
1            white_irish    0.134038    0.193342    0.023177   -0.041405
2  white_gypsy_traveller    0.007112   -0.018605   -0.076972    0.261278
3            white_other    0.171708    0.129888    0.078792   -0.035809
4      asian_bangladeshi    0.059656    0.059498    0.108997   -0.058075


In [10]:
master_loadings.columns, master_loadings.shape

(Index(['area', 'district_code', 'total_population', 'white_total',
        'white_british', 'white_irish', 'white_gypsy_traveller', 'white_other',
        'mixed_total', 'mixed_white_black_caribbean',
        ...
        'public_admin_and_defence_prop', 'education_prop',
        'health_and_social_work_prop', 'other_services_prop', 'PCA_1', 'PCA_2',
        'PCA_3', 'PCA_4', 'TSNE_X', 'TSNE_Y'],
       dtype='str', length=122),
 (636, 122))

In [18]:
pca_sort = pca.sort_values(by="PC1_Weight")

In [12]:
pca_sort[["Unnamed: 0", "PC1_Weight"]].head()

,Unnamed: 0,PC1_Weight
36,water_and_waste_management,-0.191445
34,manufacturing,-0.189952
37,construction,-0.187358
38,wholesale_and_retail,-0.184566
48,health_and_social_work,-0.183348


In [13]:
pca_sort[["Unnamed: 0", "PC1_Weight"]].tail()

,Unnamed: 0,PC1_Weight
13,mixed_white_black_african,0.172792
17,other_remaining,0.174384
24,professional_occupations,0.177411
15,mixed_other,0.193492
12,mixed_white_asian,0.195585


In [14]:
pca_sort2 = pca.sort_values(by="PC2_Weight")

In [15]:
pca_sort2[["Unnamed: 0", "PC2_Weight"]].head()

,Unnamed: 0,PC2_Weight
20,fair_health,-0.153339
0,white_british,-0.145949
28,caring_and_leisure,-0.143131
27,skilled_trades_occupations,-0.120109
30,process_plant_and_machine_operatives,-0.118734


In [16]:
pca_sort2[["Unnamed: 0", "PC2_Weight"]].tail()

,Unnamed: 0,PC2_Weight
42,financial_and_insurance,0.213505
49,other_services,0.218572
43,real_estate_activities,0.218973
41,information_and_communication,0.233837
44,professional_and_scientific,0.235529


In [19]:
pca_sort3 = pca.sort_values(by="PC3_Weight")

In [26]:
pca_sort3[["Unnamed: 0", "PC3_Weight"]].head()

,Unnamed: 0,PC3_Weight
23,managers_directors_and_senior_officials,-0.299008
0,white_british,-0.200762
25,associate_prof_and_tech,-0.189972
18,very_good_health,-0.175444
24,professional_occupations,-0.148264


In [27]:
pca_sort3[["Unnamed: 0", "PC3_Weight"]].tail()

,Unnamed: 0,PC3_Weight
30,process_plant_and_machine_operatives,0.241217
21,bad_health,0.241957
22,very_bad_health,0.262463
29,sales_and_customer_service,0.266534
31,elementary_occupations,0.315725
